# Data-matched baselines: YOLO26s-seg and SAM 2.1 on the U-Net training set

Retrains YOLO26s-seg and fine-tunes SAM 2.1 on exactly the training data of the primary U-Net
(123 training images including 38 video crops, without the frame of the internal test recording; validation on the six
photographs). Settings are those of the paper. Runtime > Change runtime type > GPU (A100 or L4) before starting.

In [ ]:
# 1. GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Google Drive (results survive a disconnect)
from google.colab import drive
drive.mount('/content/drive')
import os
WORK = '/content/drive/MyDrive/angusclaw_matched'
os.makedirs(WORK, exist_ok=True)
print(WORK)

In [ ]:
# 3. Software: Ultralytics as in the paper, the SAM 2 code (imported from the clone, not installed)
!pip -q install ultralytics==8.4.189 pycocotools hydra-core iopath
!git clone -q https://github.com/facebookresearch/sam2 /content/pkg/sam2 2>/dev/null || echo 'sam2 already cloned'

In [ ]:
# 4. Package: put matched_baselines_package.zip into the Drive folder above, or upload it when asked
import shutil, zipfile
PKG = f'{WORK}/matched_baselines_package.zip'
if not os.path.exists(PKG):
    from google.colab import files
    uploaded = files.upload()          # choose matched_baselines_package.zip
    shutil.move(next(iter(uploaded)), PKG)
with zipfile.ZipFile(PKG) as z:
    z.extractall('/content/pkg')
# absolute dataset path for Ultralytics
open('/content/pkg/yolo_matched/data.yaml', 'w').write(
    'path: /content/pkg/yolo_matched\ntrain: images/train\nval: images/val\nnames:\n  0: claw\n')
print(len(os.listdir('/content/pkg/yolo_matched/images/train')), 'YOLO training windows (expected 156)')

In [ ]:
# 5. YOLO26s-seg, settings of the paper (configs/yolo26s_claw_seg.yaml), about 15-30 min on an A100
from ultralytics import YOLO
model = YOLO('yolo26s-seg.pt')        # COCO-pre-trained weights, downloaded automatically
model.train(cfg='/content/pkg/configs/yolo26s_claw_seg.yaml', data='/content/pkg/yolo_matched/data.yaml',
            project=f'{WORK}/runs', name='yolo26s_matched', exist_ok=True)

In [ ]:
# 6. SAM 2.1 base_plus checkpoint (official)
os.makedirs('/content/pkg/weights/SAM2.1-base', exist_ok=True)
!wget -q -nc -P /content/pkg/weights/SAM2.1-base https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt
!ls -la /content/pkg/weights/SAM2.1-base

In [ ]:
# 7. SAM 2.1 fine-tuning, settings of the paper (decoder, prompt encoder, last 4 Hiera blocks and neck, 20 epochs)
!cd /content/pkg && python scripts/finetune_sam2.py --data sam_matched --model base_plus \
    --ckpt weights/SAM2.1-base/sam2.1_hiera_base_plus.pt --train-blocks 4 --epochs 20 \
    --out {WORK}/runs/sam21_matched

In [ ]:
# 8. Package the trained models and logs (no images) and download them
import glob
zip_path = f'{WORK}/matched_baselines_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    y = f'{WORK}/runs/yolo26s_matched'
    for p in [f'{y}/weights/best.pt', f'{y}/args.yaml', f'{y}/results.csv']:
        if os.path.exists(p): z.write(p, 'yolo26s_matched/' + os.path.basename(p))
    s = f'{WORK}/runs/sam21_matched'
    for p in [f'{s}/best.pt', f'{s}/train_config.json', f'{s}/log.csv']:
        if os.path.exists(p): z.write(p, 'sam21_matched/' + os.path.basename(p))
print(zip_path, round(os.path.getsize(zip_path) / 1e6), 'MB')
from google.colab import files
files.download(zip_path)